<a href="https://colab.research.google.com/github/MaynineHuang/COMPSCI764_RAG/blob/main/notebooks/Retrieval_V1_Plain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/MaynineHuang/COMPSCI764_RAG.git
%cd COMPSCI764_RAG

fatal: destination path 'COMPSCI764_RAG' already exists and is not an empty directory.
/content/COMPSCI764_RAG


In [2]:
!pip install -q datasets sentence-transformers

In [3]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util

In [4]:
#load the data
hotpot = load_dataset(
        "hotpotqa/hotpot_qa",
        "distractor",
        split="validation[:5]"
         )
print(hotpot)

Dataset({
    features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
    num_rows: 5
})


In [5]:
#load the embedding model
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
retrieval_model = SentenceTransformer(MODEL_NAME)
print("Model loaded:", MODEL_NAME)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded: sentence-transformers/all-MiniLM-L6-v2


In [6]:
#define the retrieve_top_k
def retrieve_top_k(example, model, top_k=5):
  question = example["question"]
  titles = example["context"]["title"]
  sentence_lists = example["context"]["sentences"]

  passages = [
      title + ". " + " ".join(sentences)
      for title, sentences in zip(titles, sentence_lists)
  ]
  question_embedding = model.encode(
      question,
      convert_to_tensor=True
  )

  passage_embeddings = model.encode(
      passages,
      convert_to_tensor=True
  )

  scores = util.cos_sim(
      question_embedding,
      passage_embeddings    )[0]

  ranked_indices = scores.argsort(descending=True)
  top_indices = ranked_indices[:top_k]

  results = []

  for rank, idx in enumerate(top_indices):
    idx = int(idx)
    results.append({
        "rank": rank + 1,
        "title": titles[idx],
        "passage": passages[idx],
        "score": float(scores[idx])
        })

  return results

In [7]:
#test the function on example1
results = retrieve_top_k(
    hotpot[0],
    retrieval_model,
    top_k=5
)
for result in results:
  print(
      result["rank"],
      round(result["score"], 4),
      result["title"]
  )

1 0.552 Ed Wood
2 0.4753 Ed Wood (film)
3 0.3287 Adam Collis
4 0.3027 Conrad Brooks
5 0.2922 Woodson, Arkansas
